In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    precision_recall_curve
)

from imblearn.under_sampling import RandomUnderSampler

import joblib

In [ ]:
# ============================================================
# 1. LOAD DATASET
# ============================================================
# Uses the deduplicated dataset produced by 01_data_exploration.

df = pd.read_csv("../data/processed/creditcard_processed.csv")

print("Dataset shape:")
print(df.shape)

print("\nOriginal class distribution:")
print(df["Class"].value_counts())

print("\nDuplicate rows:")
print(df.duplicated().sum())

In [ ]:
# ============================================================
# 2. SEPARATE FEATURES AND TARGET
# ============================================================

X = df.drop("Class", axis=1)
y = df["Class"]

In [ ]:
# ============================================================
# 3. TRAIN / TEST SPLIT
# ============================================================
# IMPORTANT:
# We split BEFORE undersampling.
# The test set must remain untouched so we can compare
# every technique (SMOTE, undersampling, class weights,
# threshold tuning) on the exact same data.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
# ============================================================
# 4. CHECK TRAINING DATA BEFORE UNDERSAMPLING
# ============================================================

print("\nBefore undersampling:")
print(y_train.value_counts())

In [ ]:
# ============================================================
# 5. APPLY RANDOM UNDERSAMPLING
# ============================================================
# Undersampling is applied ONLY to the training data.
# RandomUnderSampler discards random majority (legitimate)
# samples until both classes are balanced.
# The test data is NOT modified.

rus = RandomUnderSampler(
    random_state=42
)

X_train_under, y_train_under = rus.fit_resample(
    X_train,
    y_train
)

In [ ]:
# ============================================================
# 6. CHECK TRAINING DATA AFTER UNDERSAMPLING
# ============================================================

print("\nAfter undersampling:")
print(y_train_under.value_counts())

print("\nTraining size after undersampling:")
print(len(y_train_under))

In [ ]:
# ============================================================
# 7. TRAIN GRADIENT BOOSTING CLASSIFIER
# ============================================================
# Same model as the SMOTE notebook so results are comparable.
# Undersampling shrinks the dataset, so training is fast.

model = GradientBoostingClassifier(
    random_state=42
)

print("\nTraining Gradient Boosting model...")

model.fit(
    X_train_under,
    y_train_under
)

print("Model training completed!")

In [ ]:
# ============================================================
# 8. PREDICT FRAUD PROBABILITIES
# ============================================================
# We use probabilities instead of directly using predict()
# because probability thresholds are important for fraud detection.

y_prob = model.predict_proba(X_test)[:, 1]

print("\nFirst 10 fraud probabilities:")
print(y_prob[:10])

In [ ]:
# ============================================================
# 9. CLASSIFY USING THRESHOLD = 0.5
# ============================================================

threshold = 0.5

y_pred = (y_prob >= threshold).astype(int)

In [ ]:
# ============================================================
# 10. CALCULATE EVALUATION METRICS
# ============================================================

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Average Precision is commonly used as a summary
# measure for the Precision-Recall curve.
pr_auc = average_precision_score(y_test, y_prob)

In [ ]:
# ============================================================
# 11. PRINT RESULTS
# ============================================================

print("\n================================")
print("   UNDERSAMPLING RESULTS")
print("================================")

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")

print("================================")

In [ ]:
# ============================================================
# 12. PRECISION-RECALL CURVE
# ============================================================

precision_curve, recall_curve, thresholds = precision_recall_curve(
    y_test,
    y_prob
)

plt.figure(figsize=(8, 6))

plt.plot(
    recall_curve,
    precision_curve,
    label=f"Undersampling (PR-AUC = {pr_auc:.4f})"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Undersampling - Precision-Recall Curve")

plt.legend()
plt.grid()

plt.show()

In [ ]:
# ============================================================
# 13. SAVE MODEL
# ============================================================

joblib.dump(model, "../models/undersampling_model.pkl")

print("Undersampling model saved successfully.")